In [ ]:
import io
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets


# --- File upload widget with proper Voila handling ---
upload = widgets.FileUpload(accept='image/*', multiple=False, description='Upload Image')


# --- Colormap selector ---
cmap_selector = widgets.Dropdown(options=plt.colormaps(), value='gray', description='Colormap:')


# --- Scaling selector ---
scale_selector = widgets.Dropdown(options=['linear', 'log'], value='log', description='FFT Scale:')


# --- FFT zoom slider ---
zoom_slider = widgets.IntSlider(value=100, min=10, max=500, step=10, description='FFT Crop:', continuous_update=True)


# --- Filter type selector ---
filter_type_selector = widgets.Dropdown(options=['None', 'Low-pass', 'High-pass'], value='None', description='Filter:')


# --- Filter radius slider ---
filter_radius_slider = widgets.IntSlider(value=50, min=5, max=250, step=5, description='Radius:', continuous_update=True)


# --- Display controls ---
controls = widgets.VBox([upload, cmap_selector, scale_selector, zoom_slider, filter_type_selector, filter_radius_slider])
display(controls)


# --- Output widgets ---
out_original = widgets.Output()
out_fft = widgets.Output()
out_filtered = widgets.Output()
display(widgets.HBox([out_original, out_fft, out_filtered]))


# --- Function to create circular filter mask ---
def circular_mask(shape, radius, filter_type):
rows, cols = shape
crow, ccol = rows//2, cols//2
Y, X = np.ogrid[:rows, :cols]
distance = np.sqrt((X - ccol)**2 + (Y - crow)**2)
if filter_type == 'Low-pass':
mask = distance <= radius
elif filter_type == 'High-pass':
mask = distance >= radius
else:
mask = np.ones(shape, dtype=bool)
return mask


# --- Function to update plots ---
def update_plot(change=None):
if upload.value:
uploaded_file = list(upload.value.values())[0]
content = uploaded_file['content']
img = Image.open(io.BytesIO(content)).convert('L')
img_array = np.array(img)


# Compute FFT
fft = np.fft.fft2(img_array)
fft_shifted = np.fft.fftshift(fft)
magnitude = np.abs(fft_shifted)
if scale_selector.value == 'log':
magnitude_display = np.log1p(magnitude)
else:
magnitude_display = magnitude.copy()


# Apply filter
mask = circular_mask(fft_shifted.shape, filter_radius_slider.value, filter_type_selector.value)
fft_filtered = fft_shifted * mask


# Crop FFT
filter_radius_slider.observe(update_plot, names='value')